In [11]:
import pandas as pd
import numpy as np
# === GENEROWANIE SYNTETYCZNEGO DATASETU CALIFORNIA HOUSING ===
def make_california_housing(n=5000, seed=42):
    """Generuje syntetyczny dataset wzorowany na California Housing."""
    rng = np.random.default_rng(seed)
    # Współrzędne geograficzne w obszarze Kalifornii (szerokość ~32-42, długość ~-124..-114)
    latitude = rng.uniform(32.5, 41.5, n)
    longitude = rng.uniform(-123.5, -114.5, n)
    # Dochód: rozkład skośny
    med_inc = rng.gamma(shape=2.0, scale=1.8, size=n) + 0.5
    # Wiek domu
    house_age = rng.integers(1, 55, n)
    # Pokoje/sypialnie/populacja
    ave_rooms = rng.normal(5.5, 1.5, n).clip(1, 15)
    ave_bedrms = (ave_rooms * rng.uniform(0.15, 0.25, n)).clip(0.5, 5)
    population = rng.integers(50, 5000, n)
    ave_occup = rng.normal(3.0, 0.8, n).clip(1, 10)
    # Cena domu (w 100_000 USD) - zależna od dochodu, wieku i lokalizacji
    med_house_val = (
    med_inc * 0.6
    + (latitude - 36) * 0.15
    - (house_age / 100)
    + rng.normal(0, 0.3, n)
    ).clip(0.15, 5.0)
    return pd.DataFrame({
        'MedInc': med_inc.round(4),
        'HouseAge': house_age,
        'AveRooms': ave_rooms.round(4),
        'AveBedrms': ave_bedrms.round(4),
        'Population': population,
        'AveOccup': ave_occup.round(4),
        'Latitude': latitude.round(4),
        'Longitude': longitude.round(4),
        'MedHouseVal': med_house_val.round(4)
    })
    df_housing = make_california_housing(n=5000, seed=42)
    print("=== Syntetyczny California Housing Dataset ===")
    print(df_housing.head())
    print(f"\nShape: {df_housing.shape}")
    print(f"\nOpis:\n{df_housing.describe().round(3)}")

In [12]:
df_housing = make_california_housing(n=2000)

In [14]:
df_housing["lat_bin"] = pd.cut(df_housing["Latitude"], 4)
df_housing["lon_bin"] = pd.cut(df_housing["Longitude"], 4)

In [15]:
df_housing[["Latitude", "lat_bin", "Longitude", "lon_bin"]].head()

,Latitude,lat_bin,Longitude,lon_bin
0,39.4656,"(39.247, 41.495]",-115.9154,"(-116.75, -114.502]"
1,36.4499,"(34.752, 37.0]",-119.4825,"(-121.246, -118.998]"
2,40.2274,"(39.247, 41.495]",-114.9232,"(-116.75, -114.502]"
3,38.7763,"(37.0, 39.247]",-117.6428,"(-118.998, -116.75]"
4,33.3476,"(32.496, 34.752]",-122.4569,"(-123.503, -121.246]"


In [17]:
df_housing["region"] = (
    df_housing["lat_bin"].astype(str) + "|" +
    df_housing["lon_bin"].astype(str)
)

In [18]:
df_housing[["lat_bin", "lon_bin", "region"]].head()

,lat_bin,lon_bin,region
0,"(39.247, 41.495]","(-116.75, -114.502]","(39.247, 41.495]|(-116.75, -114.502]"
1,"(34.752, 37.0]","(-121.246, -118.998]","(34.752, 37.0]|(-121.246, -118.998]"
2,"(39.247, 41.495]","(-116.75, -114.502]","(39.247, 41.495]|(-116.75, -114.502]"
3,"(37.0, 39.247]","(-118.998, -116.75]","(37.0, 39.247]|(-118.998, -116.75]"
4,"(32.496, 34.752]","(-123.503, -121.246]","(32.496, 34.752]|(-123.503, -121.246]"


In [19]:
region_mean = (
    df_housing
    .groupby("region")["MedHouseVal"]
    .mean()
)

In [20]:
region_mean.sort_values(ascending=False).head(5)

region
(39.247, 41.495]|(-118.998, -116.75]     2.863874
(39.247, 41.495]|(-116.75, -114.502]     2.813552
(39.247, 41.495]|(-123.503, -121.246]    2.649020
(39.247, 41.495]|(-121.246, -118.998]    2.643564
(37.0, 39.247]|(-121.246, -118.998]      2.573469
Name: MedHouseVal, dtype: float64

In [21]:
top_5 = region_mean.sort_values(ascending=False).head(5)
top_5

region
(39.247, 41.495]|(-118.998, -116.75]     2.863874
(39.247, 41.495]|(-116.75, -114.502]     2.813552
(39.247, 41.495]|(-123.503, -121.246]    2.649020
(39.247, 41.495]|(-121.246, -118.998]    2.643564
(37.0, 39.247]|(-121.246, -118.998]      2.573469
Name: MedHouseVal, dtype: float64

zadanie 5 


In [22]:
df = make_california_housing(n=2000)

In [23]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,1.6734,19,4.7507,1.1324,2645,2.9452,39.4656,-115.9154,1.4731
1,7.8686,40,2.6891,0.5991,1550,2.3519,36.4499,-119.4825,4.3589
2,2.5247,21,6.6706,1.5262,447,2.8289,40.2274,-114.9232,2.0041
3,0.7261,21,3.1792,0.5000,2361,4.4639,38.7763,-117.6428,0.2472
4,2.8233,52,3.5833,0.7914,3501,2.9136,33.3476,-122.4569,0.5558


In [24]:
C = np.corrcoef(df.values.T)

In [25]:
C

array([[ 1.        , -0.00878967,  0.01926581,  0.01177061,  0.00825595,
        -0.03343156,  0.01767043,  0.02565535,  0.90755379],
       [-0.00878967,  1.        ,  0.02563232,  0.02943039,  0.03017518,
         0.01356082, -0.00131199, -0.01913276, -0.10995291],
       [ 0.01926581,  0.02563232,  1.        ,  0.87507484, -0.03459028,
         0.04792139, -0.01911767,  0.02360194,  0.00538515],
       [ 0.01177061,  0.02943039,  0.87507484,  1.        , -0.03082111,
         0.02926402, -0.01606206,  0.02232911, -0.00564844],
       [ 0.00825595,  0.03017518, -0.03459028, -0.03082111,  1.        ,
         0.01470136,  0.0174733 , -0.01594834,  0.01164387],
       [-0.03343156,  0.01356082,  0.04792139,  0.02926402,  0.01470136,
         1.        ,  0.02380365,  0.00499435, -0.02419164],
       [ 0.01767043, -0.00131199, -0.01911767, -0.01606206,  0.0174733 ,
         0.02380365,  1.        , -0.00470459,  0.26796026],
       [ 0.02565535, -0.01913276,  0.02360194,  0.02232911, -0

In [26]:
def corr_to_char(r):
    if r >= 0.75:
        return "█"
    if r >= 0.50:
        return "▓"
    if r >= 0.25:
        return "▒"
    if r >= 0.05:
        return "░"
    if r > -0.05:
        return "."
    if r > -0.25:
        return "-"
    if r > -0.50:
        return "="
    if r > -0.75:
        return "#"
    return "@"

In [27]:
def ascii_heatmap(M, labels):
    n = len(labels)
    short = [l[:7] for l in labels]

    print("\nASCII heatmap macierzy korelacji")
    header = " " * 9 + "".join(f"{s:>8}" for s in short)
    print(header)

    for i in range(n):
        row = f"{short[i]:>8} "
        for j in range(n):
            row += f"{corr_to_char(M[i,j])} {M[i,j]:+.2f} "
        print(row)

In [28]:
ascii_heatmap(C, labels=df.columns)


ASCII heatmap macierzy korelacji
           MedInc HouseAg AveRoom AveBedr Populat AveOccu Latitud Longitu MedHous
  MedInc █ +1.00 . -0.01 . +0.02 . +0.01 . +0.01 . -0.03 . +0.02 . +0.03 █ +0.91 
 HouseAg . -0.01 █ +1.00 . +0.03 . +0.03 . +0.03 . +0.01 . -0.00 . -0.02 - -0.11 
 AveRoom . +0.02 . +0.03 █ +1.00 █ +0.88 . -0.03 . +0.05 . -0.02 . +0.02 . +0.01 
 AveBedr . +0.01 . +0.03 █ +0.88 █ +1.00 . -0.03 . +0.03 . -0.02 . +0.02 . -0.01 
 Populat . +0.01 . +0.03 . -0.03 . -0.03 █ +1.00 . +0.01 . +0.02 . -0.02 . +0.01 
 AveOccu . -0.03 . +0.01 . +0.05 . +0.03 . +0.01 █ +1.00 . +0.02 . +0.00 . -0.02 
 Latitud . +0.02 . -0.00 . -0.02 . -0.02 . +0.02 . +0.02 █ +1.00 . -0.00 ▒ +0.27 
 Longitu . +0.03 . -0.02 . +0.02 . +0.02 . -0.02 . +0.00 . -0.00 █ +1.00 . +0.02 
 MedHous █ +0.91 - -0.11 . +0.01 . -0.01 . +0.01 . -0.02 ▒ +0.27 . +0.02 █ +1.00 


In [29]:
pairs = []

for i in range(len(df.columns)):
    for j in range(i + 1, len(df.columns)):
        if df.columns[i] != "MedHouseVal" and df.columns[j] != "MedHouseVal":
            pairs.append((df.columns[i], df.columns[j], abs(C[i, j])))

pairs = sorted(pairs, key=lambda x: x[2], reverse=True)

pairs[:3]

[('AveRooms', 'AveBedrms', np.float64(0.8750748428616181)),
 ('AveRooms', 'AveOccup', np.float64(0.047921386221286784)),
 ('AveRooms', 'Population', np.float64(0.03459028318875515))]

Najsilniejszą korelację wykazują cechy `AveRooms` i `AveBedrms`
(r ≈ 0.875). Co się zgadza,  ponieważ średnia liczba sypialni
jest generowana na podstawie średniej liczby pokoi.
Pozostałe dwie pary z top-3 mają bardzo słabą korelację:
`AveRooms` i `AveOccup` (r ≈ 0.048) oraz `AveRooms` i
`Population` (r ≈ 0.035). Oznacza to, że w tym syntetycznym
zbiorze zależność liniowa między tymi cechami jest praktycznie
nieistotna.